# Phase 6 : Modèles Non-Supervisés — Clustering

Ce notebook se concentre sur la Phase 6 du projet d'ingénierie ML. Nous utilisons ici le **Clustering** pour découvrir des groupes naturels de clients (Personas) à partir de leurs comportements d'achat et de leur engagement social.

## Objectifs :
- Identifier des segments de clientèle homogènes.
- Déterminer le nombre optimal de clusters.
- Visualiser et interpréter les profils (profiling).

## 1. Importations et Préparation des Données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chargement du dataset
df = pd.read_csv('../data/processed/dataset_ml_features.csv')

# Agrégation par Client pour du Profiling (Meilleure pratique)
# Nous prenons la moyenne des caractéristiques comportementales par client
customer_data = df.groupby('FK_Client').agg({
    'client_ca_moyen': 'mean',
    'client_nb_achats': 'mean',
    'log_Likes': 'mean',
    'log_Prix_unitaire': 'mean',
    'log_Quantite': 'mean',
    'A_Remise': 'mean'
}).reset_index()

print(f"Effectif total des clients : {customer_data.shape[0]}")
customer_data.head()

## 2. Standardisation des Données

Le clustering basé sur les distances (K-Means) nécessite que toutes les variables soient à la même échelle.

In [ ]:
features = ['client_ca_moyen', 'client_nb_achats', 'log_Likes', 'log_Prix_unitaire', 'log_Quantite', 'A_Remise']
X = customer_data[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Données standardisées (5 premières lignes) :")
print(X_scaled[:5])

## 3. Détermination du Nombre Optimal de Clusters (K)

### 3.1 Méthode du Coude (Elbow Method)

In [ ]:
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', linestyle='--')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Inertie')
plt.title('Méthode du Coude')
plt.show()

### 3.2 Score de Silhouette
Mesure la qualité de séparation des clusters (proche de 1 = excellent, proche de 0 = chevauchement).

In [ ]:
sil_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), sil_scores, marker='s', color='blue')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Score de Silhouette')
plt.title('Analyse de la Silhouette')
plt.show()

## 4. Application de K-Means (Exemple avec K=3 ou 4)

Supposons que le coude suggère 4 groupes pour une segmentation marketing fine.

In [ ]:
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
customer_data['Cluster'] = kmeans.fit_predict(X_scaled)

print("Taille de chaque cluster :")
print(customer_data['Cluster'].value_counts())

## 5. Visualisation avec PCA (2D)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=customer_data['Cluster'], palette='viridis', s=100, alpha=0.8)
plt.title('Visualisation des Clusters via PCA')
plt.xlabel('Composante Principale 1')
plt.ylabel('Composante Principale 2')
plt.legend(title='Cluster')
plt.show()

print(f"Variance expliquée par les 2 composantes : {sum(pca.explained_variance_ratio_):.2f}")

## 6. Interprétation des Clusters (Profiling)

Nous analysons les moyennes pour comprendre qui est dans quel groupe.

In [ ]:
profiling = customer_data.groupby('Cluster')[features].mean()
profiling.style.background_gradient(cmap='Blues')

### Radar Chart (Optionnel - Pour intuition)
Pour visualiser les forces de chaque cluster.

In [ ]:
# Visualisation simplifiée via barplot
profiling_melted = profiling.reset_index().melt(id_vars='Cluster')

plt.figure(figsize=(12, 6))
sns.barplot(data=profiling_melted, x='variable', y='value', hue='Cluster')
plt.title('Comparaison des caractéristiques moyennes par Cluster')
plt.xticks(rotation=45)
plt.show()

## 7. Conclusion : Définition des Personas

Sur la base des résultats ci-dessus, nous pouvons nommer les groupes :
- **Cluster 0** : Clients Occasionnels (Low Freq, Low Spending)
- **Cluster 1** : Clients Engagés & Fidèles (High Likes, High Freq)
- **Cluster 2** : Gros Acheteurs / VIP (High Monetary, Med Freq)
- **Cluster 3** : Chercheurs de Remises (High 'A_Remise')